# 01. Аудит данных

## Цель ноутбука

Цель этого ноутбука — проверить структуру датасета, качество данных и пригодность таблиц для построения продуктовой воронки интернет-магазина.

В рамках аудита нужно:

- загрузить все основные таблицы;
- проверить размеры и состав столбцов;
- посмотреть пропуски и дубликаты;
- изучить временные поля;
- разобрать типы событий;
- понять, можно ли надежно построить воронку на уровне сессий.

## Основной результат

По итогам ноутбука должно быть понятно:

1. какие таблицы нужны для основного анализа;
2. какие ключи используются для связи таблиц;
3. какие события войдут в воронку;
4. есть ли проблемы качества данных;
5. какие допущения нужно зафиксировать перед расчетом воронки.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

In [2]:
DATA_PATH = Path("../data/raw")

customers_path = DATA_PATH / "customers.csv"
events_path = DATA_PATH / "events.csv"
order_items_path = DATA_PATH / "order_items.csv"
orders_path = DATA_PATH / "orders.csv"
products_path = DATA_PATH / "products.csv"
reviews_path = DATA_PATH / "reviews.csv"
sessions_path = DATA_PATH / "sessions.csv"

print("Путь к данным:", DATA_PATH.resolve())
print("Файлы в папке data/raw:")
for file in sorted(DATA_PATH.glob("*.csv")):
    print("-", file.name)

Путь к данным: /Users/mikle/product-analytics-portfolio/01-funnel-diagnosis/data/raw
Файлы в папке data/raw:
- customers.csv
- events.csv
- order_items.csv
- orders.csv
- products.csv
- reviews.csv
- sessions.csv


In [3]:
customers = pd.read_csv(customers_path)
events = pd.read_csv(events_path)
order_items = pd.read_csv(order_items_path)
orders = pd.read_csv(orders_path)
products = pd.read_csv(products_path)
reviews = pd.read_csv(reviews_path)
sessions = pd.read_csv(sessions_path)

## 1. Первичный обзор таблиц

Сначала посмотрим размеры таблиц, названия столбцов и первые строки, чтобы понять структуру данных и возможные ключи для связи.

In [4]:
tables = {
    "customers": customers,
    "events": events,
    "order_items": order_items,
    "orders": orders,
    "products": products,
    "reviews": reviews,
    "sessions": sessions,
}

In [5]:
overview = pd.DataFrame({
    "table": list(tables.keys()),
    "rows": [df.shape[0] for df in tables.values()],
    "columns": [df.shape[1] for df in tables.values()]
}).sort_values("rows", ascending=False)

overview

,table,rows,columns
1,events,760958,10
6,sessions,120000,6
2,order_items,59163,5
3,orders,33580,10
0,customers,20000,7
5,reviews,10780,6
4,products,1197,6


In [6]:
for name, df in tables.items():
    print("=" * 80)
    print(f"Таблица: {name}")
    print(f"Размер: {df.shape}")
    print("Столбцы:")
    print(df.columns.tolist())
    print()

Таблица: customers
Размер: (20000, 7)
Столбцы:
['customer_id', 'name', 'email', 'country', 'age', 'signup_date', 'marketing_opt_in']

Таблица: events
Размер: (760958, 10)
Столбцы:
['event_id', 'session_id', 'timestamp', 'event_type', 'product_id', 'qty', 'cart_size', 'payment', 'discount_pct', 'amount_usd']

Таблица: order_items
Размер: (59163, 5)
Столбцы:
['order_id', 'product_id', 'unit_price_usd', 'quantity', 'line_total_usd']

Таблица: orders
Размер: (33580, 10)
Столбцы:
['order_id', 'customer_id', 'order_time', 'payment_method', 'discount_pct', 'subtotal_usd', 'total_usd', 'country', 'device', 'source']

Таблица: products
Размер: (1197, 6)
Столбцы:
['product_id', 'category', 'name', 'price_usd', 'cost_usd', 'margin_usd']

Таблица: reviews
Размер: (10780, 6)
Столбцы:
['review_id', 'order_id', 'product_id', 'rating', 'review_text', 'review_time']

Таблица: sessions
Размер: (120000, 6)
Столбцы:
['session_id', 'customer_id', 'start_time', 'device', 'source', 'country']



In [7]:
for name, df in tables.items():
    print("=" * 80)
    print(f"Таблица: {name}")
    display(df.head(3))

Таблица: customers


,customer_id,name,email,country,age,signup_date,marketing_opt_in
0,1,Jennifer Salinas,nicholas59@example.org,JP,71,2020-09-04,True
1,2,Phillip Ramos,christinarubio@example.com,IN,26,2020-04-05,False
2,3,Dawn Fowler,jessica03@example.org,BR,21,2023-08-31,True


Таблица: events


,event_id,session_id,timestamp,event_type,product_id,qty,cart_size,payment,discount_pct,amount_usd
0,1,1,2021-12-27T00:08:36,page_view,93.00,NaN,NaN,NaN,NaN,NaN
1,2,1,2021-12-27T00:16:36,page_view,"1,005.00",NaN,NaN,NaN,NaN,NaN
2,3,1,2021-12-27T00:18:01,add_to_cart,"1,005.00",1.00,NaN,NaN,NaN,NaN


Таблица: order_items


...

,product_id,category,name,price_usd,cost_usd,margin_usd
0,1,Electronics,SSD MediumBlue 149,570.28,352.69,217.59
1,2,Electronics,Keyboard DeepPink 696,498.13,263.13,235.00
2,3,Electronics,Headphones Orchid 188,548.53,309.60,238.93


Таблица: reviews


,review_id,order_id,product_id,rating,review_text,review_time
0,1,4,1157,2,Not great quality.,2022-02-01T00:00:00
1,2,7,405,4,Good value for money!,2020-03-07T00:00:00
2,3,7,487,5,"Excellent product, highly recommend!",2024-12-26T00:00:00


Таблица: sessions


,session_id,customer_id,start_time,device,source,country
0,1,12360,2021-12-27T00:01:36,mobile,email,DE
1,2,13917,2025-01-31T21:29:42,desktop,organic,PL
2,3,1022,2024-02-19T00:52:50,tablet,organic,FR


## 2. Типы данных и пропуски

На этом этапе нужно понять:

- какие столбцы надо привести к datetime;
- есть ли пропуски в ключевых полях;
- какие таблицы уже выглядят чисто, а какие потребуют подготовки.

In [8]:
for name, df in tables.items():
    print("=" * 80)
    print(f"Типы данных: {name}")
    print(df.dtypes)
    print()

Типы данных: customers
customer_id          int64
name                object
email               object
country             object
age                  int64
signup_date         object
marketing_opt_in      bool
dtype: object

Типы данных: events
event_id          int64
session_id        int64
timestamp        object
event_type       object
product_id      float64
qty             float64
cart_size       float64
payment          object
discount_pct    float64
amount_usd      float64
dtype: object

Типы данных: order_items
order_id            int64
product_id          int64
unit_price_usd    float64
quantity            int64
line_total_usd    float64
dtype: object

Типы данных: orders
order_id            int64
customer_id         int64
order_time         object
payment_method     object
discount_pct        int64
subtotal_usd      float64
total_usd         float64
country            object
device             object
source             object
dtype: object

Типы данных: products
product_id 

In [9]:
missing_summary = []

for name, df in tables.items():
    missing = df.isna().sum()
    missing_pct = (df.isna().mean() * 100).round(2)
    
    temp = pd.DataFrame({
        "table": name,
        "column": df.columns,
        "missing_count": missing.values,
        "missing_pct": missing_pct.values
    })
    missing_summary.append(temp)

missing_summary = pd.concat(missing_summary, ignore_index=True)
missing_summary.sort_values(["missing_pct", "missing_count"], ascending=False).head(50)

,table,column,missing_count,missing_pct
14,events,payment,727378,95.59
15,events,discount_pct,727378,95.59
16,events,amount_usd,727378,95.59
13,events,cart_size,716049,94.10
12,events,qty,617832,81.19
11,events,product_id,78489,10.31
0,customers,customer_id,0,0.00
1,customers,name,0,0.00
2,customers,email,0,0.00
3,customers,country,0,0.00


In [10]:
key_columns_to_check = {
    "customers": ["customer_id"],
    "events": ["event_id", "session_id", "timestamp", "event_type"],
    "order_items": ["order_id", "product_id"],
    "orders": ["order_id", "customer_id", "order_time"],
    "products": ["product_id"],
    "reviews": ["review_id", "order_id", "product_id"],
    "sessions": ["session_id", "customer_id", "start_time"]
}

for name, cols in key_columns_to_check.items():
    df = tables[name]
    print("=" * 80)
    print(f"Ключевые поля: {name}")
    display(df[cols].isna().sum().to_frame("missing_count"))

Ключевые поля: customers


,missing_count
customer_id,0


Ключевые поля: events


,missing_count
event_id,0
session_id,0
timestamp,0
event_type,0


Ключевые поля: order_items


...

,missing_count
product_id,0


Ключевые поля: reviews


,missing_count
review_id,0
order_id,0
product_id,0


Ключевые поля: sessions


,missing_count
session_id,0
customer_id,0
start_time,0


## 3. Проверка дубликатов

Для продуктового анализа важно понять, нет ли дубликатов на уровне основных идентификаторов:

- `customer_id`
- `session_id`
- `event_id`
- `order_id`
- `review_id`

Также отдельно проверим полные дубликаты строк.

In [11]:
id_checks = {
    "customers": "customer_id",
    "events": "event_id",
    "orders": "order_id",
    "products": "product_id",
    "reviews": "review_id",
    "sessions": "session_id"
}

duplicate_report = []

for name, id_col in id_checks.items():
    df = tables[name]
    duplicate_report.append({
        "table": name,
        "id_column": id_col,
        "duplicated_ids": df[id_col].duplicated().sum(),
        "full_row_duplicates": df.duplicated().sum()
    })

pd.DataFrame(duplicate_report)

,table,id_column,duplicated_ids,full_row_duplicates
0,customers,customer_id,0,0
1,events,event_id,0,0
2,orders,order_id,0,0
3,products,product_id,0,0
4,reviews,review_id,0,0
5,sessions,session_id,0,0


## 4. Приведение временных полей

Для корректного анализа воронки и построения временных срезов нужно привести все временные столбцы к формату datetime.

In [12]:
customers["signup_date"] = pd.to_datetime(customers["signup_date"], errors="coerce")
events["timestamp"] = pd.to_datetime(events["timestamp"], errors="coerce")
orders["order_time"] = pd.to_datetime(orders["order_time"], errors="coerce")
reviews["review_time"] = pd.to_datetime(reviews["review_time"], errors="coerce")
sessions["start_time"] = pd.to_datetime(sessions["start_time"], errors="coerce")

In [13]:
date_columns = {
    "customers": "signup_date",
    "events": "timestamp",
    "orders": "order_time",
    "reviews": "review_time",
    "sessions": "start_time"
}

date_report = []

for name, col in date_columns.items():
    df = tables[name]
    date_report.append({
        "table": name,
        "date_column": col,
        "min_date": df[col].min(),
        "max_date": df[col].max(),
        "missing_dates": df[col].isna().sum()
    })

pd.DataFrame(date_report)

,table,date_column,min_date,max_date,missing_dates
0,customers,signup_date,2020-01-01 00:00:00,2025-10-31 00:00:00,0
1,events,timestamp,2020-01-01 00:32:18,2025-11-01 01:50:04,0
2,orders,order_time,2020-01-01 01:10:58,2025-10-31 22:59:41,0
3,reviews,review_time,2020-01-01 00:00:00,2025-11-01 00:00:00,0
4,sessions,start_time,2020-01-01 00:06:58,2025-10-31 23:34:11,0


## 5. Проверка структуры событий

Таблица `events` — центральная для построения воронки, поэтому нужно отдельно проверить:

- какие значения принимает `event_type`;
- как часто встречается каждый тип события;
- есть ли странные или редкие значения;
- все ли шаги воронки действительно присутствуют в данных.

In [14]:
events["event_type"].value_counts(dropna=False)

event_type
page_view      539343
add_to_cart    143126
checkout        44909
purchase        33580
Name: count, dtype: int64

In [15]:
event_summary = events.groupby("event_type").agg(
    events_cnt=("event_id", "count"),
    sessions_cnt=("session_id", "nunique"),
    products_cnt=("product_id", "nunique")
).reset_index().sort_values("events_cnt", ascending=False)

event_summary

,event_type,events_cnt,sessions_cnt,products_cnt
2,page_view,539343,120000,1197
0,add_to_cart,143126,81518,1197
1,checkout,44909,44909,0
3,purchase,33580,33580,0


In [16]:
events_per_session = events.groupby("session_id")["event_id"].count()

events_per_session.describe()

count   120,000.00
mean          6.34
std           3.42
min           1.00
25%           3.00
50%           6.00
75%           9.00
max          17.00
Name: event_id, dtype: float64

## 6. Проверка связности таблиц

Перед построением воронки важно проверить, насколько хорошо таблицы стыкуются друг с другом по ключам:

- все ли `session_id` из `events` есть в `sessions`;
- все ли `customer_id` из `sessions` есть в `customers`;
- все ли `customer_id` из `orders` есть в `customers`;
- все ли `product_id` из `events` и `order_items` есть в `products`.

In [17]:
events_sessions_missing = set(events["session_id"].dropna()) - set(sessions["session_id"].dropna())
sessions_customers_missing = set(sessions["customer_id"].dropna()) - set(customers["customer_id"].dropna())
orders_customers_missing = set(orders["customer_id"].dropna()) - set(customers["customer_id"].dropna())
events_products_missing = set(events["product_id"].dropna()) - set(products["product_id"].dropna())
order_items_products_missing = set(order_items["product_id"].dropna()) - set(products["product_id"].dropna())

link_report = pd.DataFrame([
    {"check": "session_id из events отсутствуют в sessions", "missing_count": len(events_sessions_missing)},
    {"check": "customer_id из sessions отсутствуют в customers", "missing_count": len(sessions_customers_missing)},
    {"check": "customer_id из orders отсутствуют в customers", "missing_count": len(orders_customers_missing)},
    {"check": "product_id из events отсутствуют в products", "missing_count": len(events_products_missing)},
    {"check": "product_id из order_items отсутствуют в products", "missing_count": len(order_items_products_missing)},
])

link_report

,check,missing_count
0,session_id из events отсутствуют в sessions,0
1,customer_id из sessions отсутствуют в customers,0
2,customer_id из orders отсутствуют в customers,0
3,product_id из events отсутствуют в products,0
4,product_id из order_items отсутствуют в products,0


## 7. Проверка готовности к построению воронки

На этом этапе нужно проверить:

- сколько уникальных сессий есть в `sessions`;
- сколько сессий имеют хотя бы одно событие;
- сколько сессий проходят отдельные шаги воронки;
- можно ли использовать `events` как надежный источник шагов воронки.

In [18]:
funnel_step_map = {
    "page_view": "view_flag",
    "add_to_cart": "cart_flag",
    "checkout": "checkout_flag",
    "purchase": "purchase_flag"
}

events["event_type_norm"] = events["event_type"].astype(str).str.strip().str.lower()

session_funnel_flags = (
    events.assign(step_flag=1)
    .pivot_table(
        index="session_id",
        columns="event_type_norm",
        values="step_flag",
        aggfunc="max",
        fill_value=0
    )
    .reset_index()
)

for raw_col, new_col in funnel_step_map.items():
    if raw_col in session_funnel_flags.columns:
        session_funnel_flags[new_col] = session_funnel_flags[raw_col]
    else:
        session_funnel_flags[new_col] = 0

session_funnel_flags = session_funnel_flags[
    ["session_id", "view_flag", "cart_flag", "checkout_flag", "purchase_flag"]
]

session_funnel_flags.head()

event_type_norm,session_id,view_flag,cart_flag,checkout_flag,purchase_flag
0,1,1,1,0,0
1,2,1,1,1,1
2,3,1,1,1,1
3,4,1,0,0,0
4,5,1,0,0,0


In [19]:
sessions_base = sessions[["session_id"]].drop_duplicates().copy()

funnel_readiness = (
    sessions_base
    .merge(session_funnel_flags, on="session_id", how="left")
    .fillna(0)
)

funnel_readiness[["view_flag", "cart_flag", "checkout_flag", "purchase_flag"]] = (
    funnel_readiness[["view_flag", "cart_flag", "checkout_flag", "purchase_flag"]]
    .astype(int)
)

pd.DataFrame({
    "metric": [
        "Всего сессий в sessions",
        "Сессий с page_view",
        "Сессий с add_to_cart",
        "Сессий с checkout",
        "Сессий с purchase"
    ],
    "value": [
        funnel_readiness["session_id"].nunique(),
        funnel_readiness["view_flag"].sum(),
        funnel_readiness["cart_flag"].sum(),
        funnel_readiness["checkout_flag"].sum(),
        funnel_readiness["purchase_flag"].sum()
    ]
})

,metric,value
0,Всего сессий в sessions,120000
1,Сессий с page_view,120000
2,Сессий с add_to_cart,81518
3,Сессий с checkout,44909
4,Сессий с purchase,33580


## 8. Предварительные выводы

После выполнения всех проверок в этом разделе нужно коротко зафиксировать:

- какие таблицы являются основными для первого кейса;
- какие ключи используются для связи;
- какие события войдут в воронку;
- есть ли критичные проблемы качества данных;
- можно ли переходить к построению основной воронки.